# rStar (Self-play muTuAl Reasoning)

## Components:
1. MCTS --> Smart gambler (could increase simulations per iteration to build confidence)
2. Beam Search --> Focuses the race, keeping the `k` most promising solution trajectories at each step
3. Rich Set of Reasoning Actions --> Promote human like decision making
4. Mutual Consistency -->  Peer2peer feedback (if you and your friend got the same answer. Then you have a higher confidence you are both right)
5. Tree Nodes --> Act as a logbook entry (stores all the vital information about a specific point in the problem solving process)
6. LLM --> Powerful AI consultant/expert
7. Smart judge --> Checks if 2 outputs are equivalent (0.5 vs 1/2 vs \frac{1}{2})

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os
# Set up LLM connection

load_dotenv()
groq_key = os.getenv("GROQ_KEY", "Empty")

model = getpass("Enter the model name: ")
api_endpoint = getpass("Enter the API endpoint (default: https://api.openai.com): ")
#- Model should be "llama3-8b-8192"
#- endpoint should be "https://api.groq.com/openai"

api_endpoint = api_endpoint if api_endpoint else "https://api.openai.com"
api_key = groq_key

openai_api_base = f"{api_endpoint}/v1"

print(f"Model: {model}")
print(f"API Endpoint: {api_endpoint}")
print(f"OpenAI API Base: {openai_api_base}")
if api_key == "Empty":
    print("No API key needed.")
else:
    print(f"API Key Set")

Model: llama3-8b-8192
API Endpoint: https://api.groq.com/openai
OpenAI API Base: https://api.groq.com/openai/v1
API Key Set


In [2]:
from openai import OpenAI
import re # support for regex

# Initialize the OpenAI API client
client = OpenAI(
    api_key=api_key,
    base_url=openai_api_base
)

def chat_completion_request_openai(prompt): # function that will call the OpenAI API
    messages = [
        {"role": "user", "content": prompt}
    ]
    # Create chat completions using the OpenAI client
    chat_response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1.0,
        max_tokens=1500,
    )

    # Extract the completion text from the response
    if chat_response.choices:
        completion_text = chat_response.choices[0].message.content
    else:
        completion_text = "No response from the model."
    return completion_text

# Test the endpoint
if __name__ == "__main__":
    # Test the function
    prompt = "Return an array of string, where each string is a season"
    chat_response = chat_completion_request_openai(prompt)
    print(f"Prompt: {prompt}\n")
    print(f"Response: {chat_response}")

Prompt: Return an array of string, where each string is a season

Response: Here is an array of strings, where each string represents a season:

`["Spring", "Summer", "Autumn", "Winter"]`

Let me know if you'd like me to reshape or manipulate this array in some way!


In [ ]:
# Create prompt based on provided Action
def create_prompt(question, state, action):
    prompts = {
        "A1": f"Question: {question}\n"
    f"Existing Reasoning Steps: {state}\n"
    "Please propose the next one-step thought to advance the reasoning process. "
    "Focus only on the immediate next step without solving the entire problem. "
    "Ensure the step is logically consistent with the existing reasoning steps. "
    "Do not provide the final answer or skip steps. Unless the next step clearly results in the final answer. "
    "Think step by step and provide only one reasoning step.",
        "A2": f"Question: {question}\n"
    f"Existing Reasoning Steps: {state}\n"
    "Please propose the remaining reasoning steps to solve the problem completely. "
    "Think step by step and ensure logical consistency with the existing reasoning steps. "
    "Provide the final answer at the end of the reasoning process. "
    "Do not skip steps or provide incomplete reasoning.",
        "A3": f"Question: {question}\n"
    f"Existing Reasoning Steps: {state}\n"
    "Please propose the next sub-question to simplify the problem further. "
    "After proposing the sub-question, provide its answer. "
    "Ensure the sub-question logically follows from the existing reasoning steps. "
    "Do not solve the entire problem or skip intermediate sub-questions. "
    "Think step by step and provide only one sub-question and its answer.",
        "A4": f"Sub-Question: {question}\n"
    f"Original Answer: {state}\n"
    "The original answer might be incorrect. "
    "Please re-answer the sub-question using few-shot chain-of-thought reasoning. "
    "Think step by step and ensure logical consistency in your reasoning. "
    "Provide a detailed explanation and a verified final answer. "
    "Do not reference the original answer in your response.",
        "A5": f"Original Question: {question}\n"
    "The original question might be misunderstood or unclear. "
    "Please rephrase the question to make it simpler and easier to understand. "
    "Clearly list all conditions and constraints provided in the problem statement. "
    "Ensure that no information is lost or altered during the rephrasing process. "
    "Do not solve the question or provide an answer.",
    }
    prompt = prompts[action] + "\n\nIf you determine the final answer, explicitly state 'The final answer is [your numeric answer]' at the end of your response."
    return prompt


In [ ]:
import re

# Simulate a Node then rate the response (used to backpropagate the reward of an expanded node)
def simulate_then_rate(question, state, expected_answer):
    prompt = (
        f"Question: {question}\n"
        f"Existing Reasoning Steps: {state}\n"
        "Please propose the remaining reasoning steps to solve the problem completely. "
        "Think step by step and ensure logical consistency with the existing reasoning steps. "
        "Provide the final answer at the end of the reasoning process. "
        "Do not skip steps or provide incomplete reasoning."
    )
    simulated_answer = chat_completion_request_openai(prompt)
    print(f"Simulated Answer:\n{simulated_answer}\n")

    print(f"Total state:\n{state }\n\n{simulated_answer}\n")

    rating_prompt = (
        f"Question: {question}\n"
        f"Answer: {state}\n\n{simulated_answer}\n"
        f"Ground-truth Answer: {expected_answer}\n"
        "As an expert on this topic, please provide a detailed critique of the answer. "
        "Rate the answer based on correctness, completeness, and logical consistency. "
        "First state whether the answer is correct or incorrect. "
        "Provide only a critique, not a suggested answer. "
        "Then, rate the answer on a scale of 0 to 100. "
        "The response should be in the following format:\n"
        "Critique: <detailed critique>\n"
        "Rating: <rating>\n"
    )
    rating_response = chat_completion_request_openai(rating_prompt)
    print(f"Rating response:\n{rating_response}\n")

    # Extract the rating
    try:
        match = re.search(r"Rating:\s*(\d+)", rating_response) # Extract rating to be used in the UCT calculation
        if match:
            rating = int(match.group(1))
            if rating > 95: # Paper limits the rating to a maximum of 95 (maybe 96+ causes poor MCTS performance)
                rating = 95
            rating = float(rating)/100
        else:
            raise ValueError("Rating not found in the response.")
    except Exception as e:
        print(f"Error extracting rating: {e}")
        print(f"Rating response was: {rating_response}")
        rating = 0

    return rating # Simulated trajectory is not saved, only the rating is used to backpropagate the reward

# Test the function
if __name__ == "__main__":
    # Test the function
    question = "Solve the system of linear equations:\n\nx + 2y = 13\n3x - 4y = -18"
    ground_truth_answer = "x = 1.6, y = 5.7"
    # state = (
    #     "To solve the system of linear equations, we can use substitution or elimination methods.\n\n"
    #     "First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:\n"
    #     "(3)(x + 2y) = (3)(13)\n"
    #     "(1)(3x - 4y) = (1)(-18)"
    # )
    state = (
        "To solve the system of linear equations, we can use substitution or elimination methods.\n\n"
        "First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:\n"
        "(3)(x + 2y) = (3)(13)\n"
        "(1)(3x - 4y) = (1)(-18)\n\n"
        "Now, we will subtract the second equation from the first equation to eliminate the x variable:\n"
        "(3x + 6y) - (3x - 4y) = 39 - (-18)\n\n"
        "This simplifies to:\n"
        "10y = 57"
    )
    rating = simulate_then_rate(question, state, ground_truth_answer)
    print(f"Rating: {rating}")

    # Ground-truth answer to the above question:
    # x = 1.6
    # y = 5.7

    # This is working as expected.

Simulated Answer:
Now that we have eliminated the x variable, we can solve for y:

10y = 57

To solve for y, we can divide both sides of the equation by 10:

y = 57 / 10

y = 5.7

Now that we have found y, we can substitute y into one of the original equations to find the value of x. We will use the first equation:

x + 2y = 13

Substitute y = 5.7:

x + 2(5.7) = 13

x + 11.4 = 13

Subtract 11.4 from both sides:

x = 1.6

Now that we have found the values of x and y, we can write the final solution as:

(x, y) = (1.6, 5.7)

Therefore, the system of linear equations has been solved.

Total state:
To solve the system of linear equations, we can use substitution or elimination methods.

First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:
(3)(x + 2y) = (3)(13)
(1)(3x - 4y) = (1)(-18)

Now, we will subtract the second equation from the first equation to eliminate the x variable:
(3x + 6y) - (3x - 4y) = 39 - (-18)

In [ ]:
import math
import random
import numpy as np

max_children = 4

class Node:
    def __init__(self, question, state, action=None, parent=None):
        self.state = state
        self.action = action # Action taken to reach this node
        self.parent = parent
        self.original_question = question
        self.current_question = question
        self.is_answered = False
        self.children = []
        self.visits = 0
        self.value = 0

    def is_fully_expanded(self):
        # Check if the node has reached the maximum number of children or if the answer has been found
        return len(self.children) >= max_children or self.is_answered
    
    def best_child(self, exploration_weights=1.41):
        choices_weights = []
        for child in self.children:
            if child.visits == 0:
                weight = float('inf') # Prioritize unexplored nodes
            else:
                weight = (child.value / child.visits) + exploration_weights * math.sqrt(math.log(self.visits) / child.visits) # UCT calculation
                # Exploration term --> (child.value / child.visits) --> basically the average reward of the child
                # Exploitation term --> exploration_weights * math.sqrt(2 * math.log(self.visits) / child.visits) --> will be very high for unexplored nodes which will encourage exploration down that path
            choices_weights.append(weight)
        return self.children[np.argmax(choices_weights)]
    
    def most_visited_child(self): # This is used to pull the best trajectory from the MCTS tree once its generation is done (best trajectory == most likely to be the correct answer)
        return max(self.children, key=lambda child: child.visits) # Return the child with the most visits
    
    def add_child(self, child_node): # Utility function used to expand the tree
        self.children.append(child_node)

class rStar:
    def __init__(self, question, num_rollouts=1, max_depth=3, iterations=2):
        self.root = Node(question, "Begin answering the question")
        self.question = question
        self.num_rollouts = num_rollouts
        self.max_depth = max_depth
        self.iterations = iterations # Need to bound process by max_depth later

    def search(self):
        for i in range (self.iterations):
            print(f"Iteration {i+1}/{self.iterations}")
            node = self.select(self.root)
            print(f"Selected node: {node.state}")
            if not node.is_fully_expanded and not node.is_answered:
                node = self.expand(node)
                print(f"Expanded node: {node.state}")
            reward = self.simulate(node)
            print(f"Simulated reward: {reward}")
            self.backpropagate(node, reward)
        print(f"Visits to most visited child: {self.root.most_visited_child().visits}")
        return self.root.most_visited_child().state

    def select(self, node):
        while node.is_fully_expanded() and node.children:
            node = node.best_child() # Must check that the node is not answered in the best_child function
        return node
    
    def expand(self, node):
        # Find valid actions given the current node
        actions = ["A1", "A2", "A3", "A4", "A5"]
        # Filter out invalid actions and actions that have already been taken

        # Call create_prompt for each valid action
        # Expand every valid action

        # return a random child, which will be simulated
    



